# 05 — Сравнение прогонов

Ноутбук отвечает на вопрос, ради которого делается работа: **стало ли лучше и на сколько**. Ноутбук 04 описывает один эксперимент; здесь их два и больше.

Ноутбук **тонкий** (§7): логика — в `src/viz/compare.py`.

**Предусловие:** для каждого сравниваемого эксперимента прогнана Стадия 3 (есть `scores.parquet` и `eval_metrics.json`), и все они оценивались на **одном и том же тесте** — иначе сравниваются разные задачи.

Про пересчёт (§6). Разность EER и её интервал — межэкспериментные величины, которых не считает ни одна Стадия 3: каждый её прогон знает только про себя. Поэтому они считаются здесь, но правило «числа не расходятся» держится проверкой: `align_scores` сверяет EER, пересчитанный из выгрузки, с числом из `eval_metrics.json` и падает при расхождении.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import Paths
from src.viz import compare as C
from src.viz.style import apply_style, save_figure

apply_style()
PATHS = Paths(root=ROOT)

#: Сравниваемые эксперименты = поля `name` из experiments/<exp>.yaml.
#: Первый по соглашению — базовый: относительно него считаются дельты.
EXPS = ["smoke_mini"]
BASE, NEW = (EXPS[0], EXPS[-1])


## Сводная таблица

Строка на пару (эксперимент, датасет). Сравниваются строки **внутри одного датасета**: между датасетами числа несопоставимы — разные домены и разные классовые пропорции (§8).

⚠️ Пересекающиеся CI двух экспериментов **не** означают, что разницы нет. Про разницу отвечает таблица ΔEER ниже, где интервал построен у самой разности.

In [ ]:
table = C.compare_table(EXPS, paths=PATHS)
table


In [ ]:
fig = C.plot_eer_compare(EXPS, paths=PATHS)
save_figure(fig, "compare_eer", paths=PATHS)  # вектор (§7)


## ΔEER с парным бутстрэпом

Главное число работы. `ΔEER = EER(новый) − EER(базовый)`, отрицательное значение — улучшение. Интервал строится парным бутстрэпом: в каждом повторе пересэмплируется **один** набор записей, и по нему считаются оба EER, поэтому общая часть разброса (какие записи попали в тест, насколько они трудные) при вычитании сокращается.

Колонка «значимо» = интервал целиком по одну сторону нуля.

Оговорка для текста работы (§8): интервал покрывает конечность **теста**, но не случайность обучения. Это сравнение двух конкретных обученных моделей, а не двух методов вообще; последнее потребовало бы прогонов с разными seed.

In [ ]:
if len(EXPS) < 2:
    print("нужно минимум два эксперимента в EXPS")
else:
    delta = C.delta_table(BASE, NEW, paths=PATHS, n_boot=1000)
    display(delta)


## DET-кривые на одном датасете

EER — одна точка кривой. Два метода с равным EER могут по-разному вести себя в области малых ошибок, где и работает реальная система (§8). Кривая ниже и левее — лучше.

In [ ]:
for ds in C.compare_table(EXPS, paths=PATHS)["dataset_id"].unique():
    fig = C.plot_det_compare(EXPS, ds, paths=PATHS)
    save_figure(fig, f"compare_det_{ds}", paths=PATHS)


## Сравнение по разрезам

Улучшение равномерное — или метод просто выучил одно семейство атак, просев на остальных? Строится по тем группам, которые посчитаны во всех сравниваемых прогонах.

In [ ]:
for ds in C.compare_table(EXPS, paths=PATHS)["dataset_id"].unique():
    for column in ("attack_type", "condition", "vocoder"):
        try:
            fig = C.plot_group_compare(EXPS, ds, column, paths=PATHS)
        except ValueError as e:
            print(f"{ds}/{column}: {e}")
            continue
        save_figure(fig, f"compare_{ds}_by_{column}", paths=PATHS)
